<a href="https://colab.research.google.com/github/vikassinngh123/AI-ML-Learning/blob/main/06-Deep-Learning/01-PyTorch/01-PyTorch-Basic-Models/02_binary_classification_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import torch
from torch import nn

In [2]:
from sklearn.datasets import load_breast_cancer

cancer=load_breast_cancer()
X,y=cancer['data'],cancer['target']

In [3]:
X[:1]  # X has 30 features

array([[1.799e+01, 1.038e+01, 1.228e+02, 1.001e+03, 1.184e-01, 2.776e-01,
        3.001e-01, 1.471e-01, 2.419e-01, 7.871e-02, 1.095e+00, 9.053e-01,
        8.589e+00, 1.534e+02, 6.399e-03, 4.904e-02, 5.373e-02, 1.587e-02,
        3.003e-02, 6.193e-03, 2.538e+01, 1.733e+01, 1.846e+02, 2.019e+03,
        1.622e-01, 6.656e-01, 7.119e-01, 2.654e-01, 4.601e-01, 1.189e-01]])

In [4]:
y[:20]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1])

In [5]:
X.shape , y.shape

((569, 30), (569,))

In [6]:
X=torch.from_numpy(X.astype(np.float32))
y=torch.from_numpy(y.astype(np.float32))

In [7]:
X.dtype , y.dtype

(torch.float32, torch.float32)

In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [9]:
#
device="cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [12]:
class cancermodel1(nn.Module):
  def __init__(self):
    super().__init__()

    self.layer1=nn.Linear(in_features=30,out_features=64)    # this layer take 30 feature and upscale it to 64 features
    self.layer2=nn.Linear(in_features=64,out_features=32)    # takes 64 features from the previous layer and downscale it to 32 features
    self.layer3=nn.Linear(in_features=32,out_features=16)    # take 32 features from layer2 and outputs 16 features
    self.layer4=nn.Linear(in_features=16,out_features=1)     # take 16 features from layer3 and outputs a single features

    self.relu=nn.ReLU()
    #self.sigmoid=nn.Sigmoid()  #(we are using BCEWithLogitsLoss() that combines a sigmoid layer and BCEloss together)   # convert our ans in a range between 0-1

  def forward(self,x):
    return self.layer4(self.relu((self.layer3(self.relu(self.layer2(self.relu(self.layer1(x))))))))


model_1=cancermodel1().to(device)
model_1

cancermodel1(
  (layer1): Linear(in_features=30, out_features=64, bias=True)
  (layer2): Linear(in_features=64, out_features=32, bias=True)
  (layer3): Linear(in_features=32, out_features=16, bias=True)
  (layer4): Linear(in_features=16, out_features=1, bias=True)
  (relu): ReLU()
)

In [13]:
loss_fn=nn.BCEWithLogitsLoss()
optimizer=torch.optim.SGD(params=model_1.parameters(),lr=0.01)

In [14]:
X_train=X_train.to(device)
y_train=y_train.to(device)
X_test=X_test.to(device)
y_test=y_test.to(device)

In [15]:
def accuracy_fn(y_true,y_pred):
  correct=torch.eq(y_true,y_pred).sum().item()
  acc=correct/len(y_pred)
  return acc*100

In [16]:
epoch=1000

for epoch in range(epoch):
  model_1.train()

  y_logits=model_1(X_train).squeeze()                    #Raw preditions from the model_1
  y_preds=torch.round(torch.sigmoid(y_logits))           #Converting Raw predition to 0 or 1 though round and sigmoid funtion (raw-->sigmoid()[convert value in a range of 0-1]-->round()[round the preds to 0 or 1])

  train_accuracy=accuracy_fn(y_train,y_preds)            #Give the accuracy of the model using y_preds how?[eq()[equalite the no of time our model predited corretly]-->divided by the len of y_pred or total preditions-->multiple by 100 to convert it into percentage]

  loss=loss_fn(y_logits,y_train)                         #Calculate the loss using BCEWithLoss() and this take raw preditions

  optimizer.zero_grad()

  loss.backward()

  optimizer.step()

  model_1.eval()

  with torch.inference_mode():
    test_logits=model_1(X_test).squeeze()
    test_pred=torch.round(torch.sigmoid(test_logits))

    test_accuracy=accuracy_fn(y_test,test_pred)
    test_loss=loss_fn(test_logits.squeeze(),y_test)


  if epoch%100==0 or epoch == 999:
      print(f"Epoch: {epoch} | Loss: {loss:.5f} | Train Accuracy: {train_accuracy:.2f}% | Test Loss: {test_loss:.5f} | Test Accuracy: {test_accuracy:.2f}%")

Epoch: 0 | Loss: 10.88227 | Train Accuracy: 37.14% | Test Loss: 21.47114 | Test Accuracy: 62.28%
Epoch: 100 | Loss: 0.57448 | Train Accuracy: 78.46% | Test Loss: 0.53192 | Test Accuracy: 61.40%
Epoch: 200 | Loss: 0.44297 | Train Accuracy: 83.74% | Test Loss: 0.44247 | Test Accuracy: 78.95%
Epoch: 300 | Loss: 0.48192 | Train Accuracy: 90.11% | Test Loss: 0.37573 | Test Accuracy: 92.98%
Epoch: 400 | Loss: 0.31667 | Train Accuracy: 89.45% | Test Loss: 0.41442 | Test Accuracy: 76.32%
Epoch: 500 | Loss: 0.35551 | Train Accuracy: 84.40% | Test Loss: 0.19470 | Test Accuracy: 92.98%
Epoch: 600 | Loss: 0.30566 | Train Accuracy: 86.81% | Test Loss: 0.19512 | Test Accuracy: 92.98%
Epoch: 700 | Loss: 0.25826 | Train Accuracy: 90.33% | Test Loss: 0.16657 | Test Accuracy: 92.98%
Epoch: 800 | Loss: 0.27561 | Train Accuracy: 89.01% | Test Loss: 0.18797 | Test Accuracy: 92.11%
Epoch: 900 | Loss: 0.25352 | Train Accuracy: 90.33% | Test Loss: 0.16579 | Test Accuracy: 92.98%
Epoch: 999 | Loss: 0.22617 | T

In [18]:
print(f"Raw preditions {test_logits.squeeze()[:10]}")    #What raw predition looks likes
print(f"Converted preditions {test_pred[:10]}")

Raw preditions tensor([ -1.3813,  -9.8472,  -5.0223,   3.3823,   3.0264,  -7.3781, -10.4512,
         -2.6314,   2.1610,   1.1260], device='cuda:0')
Converted preditions tensor([0., 0., 0., 1., 1., 0., 0., 0., 1., 1.], device='cuda:0')


In [19]:
print("Predicted:", test_pred[:20])
print("Actual:   ", y_test[:20])

Predicted: tensor([0., 0., 0., 1., 1., 0., 0., 0., 1., 1., 1., 0., 1., 0., 1., 0., 1., 1.,
        1., 0.], device='cuda:0')
Actual:    tensor([1., 0., 0., 1., 1., 0., 0., 0., 1., 1., 1., 0., 1., 0., 1., 0., 1., 1.,
        1., 0.], device='cuda:0')


In [20]:
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 96.49%
